#### Let's first install and import the necessary librairies

In [76]:
import os
import pandas as pd
import numpy as np
import zipfile
import xgboost as xgb

from khiops import core as kh
from khiops.sklearn import KhiopsClassifier
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from codecarbon import EmissionsTracker


#### Let's load our data

In [98]:
# Charger le fichier de données 
fichier = '100000-vars-qq-level.txt'
donnees = pd.read_csv(fichier, sep='\t')

# Si vous avez une colonne cible, séparez-la
X1 = donnees.drop('Target', axis=1)  # Features
y = donnees['Target']                # Cible

X = X1.drop('Patient_ID', axis=1)  # Features


# Diviser les données en 90% entraînement et 10% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42
)

In [68]:
kc_model = KhiopsClassifier(n_trees=0)
kc_model.fit(X_train, y_train)
y_pred = kc_model.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print(f"Train Accuracy with Khiops: {accuracy}")

y_pred_proba = kc_model.predict_proba(X_train)[:, 1]
auc = roc_auc_score(y_train, y_pred_proba)
print(f"Train AUC with Khiops:  : {auc}")
auc_train=auc

y_pred = kc_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with Khiops: {accuracy}")

y_pred_proba = kc_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC with Khiops:  : {auc}")

r=auc/auc_train
print(f"Robustess:  : {r}")

Train Accuracy with Khiops: 0.9916243654822335
Train AUC with Khiops:  : 0.9991481041322876
Test Accuracy with Khiops: 0.9406392694063926
Test AUC with Khiops:  : 0.9784386306125437
Robustess:  : 0.9792728691231125


In [51]:
#print(f"Features evaluated: {kc_model.n_features_evaluated_}")
#print(f"Features selected : {kc_model.n_features_used_}")
#for i, feature in enumerate(kc_model.feature_used_names_[:]):
#    print(f"{feature} - Importance: {kc_model.feature_used_importances_[i][2]}")
#print("---")

In [69]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print(f"Train Accuracy with RF: {accuracy}")

y_pred_proba = rf_model.predict_proba(X_train)[:, 1]
auc = roc_auc_score(y_train, y_pred_proba)
print(f"Train AUC with RF:  : {auc}")
auc_train=auc

y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with RF: {accuracy}")

y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC with RF:  : {auc}")

r=auc/auc_train
print(f"Robustess:  : {r}")

Train Accuracy with RF: 0.9357868020304568
Train AUC with RF:  : 0.9841747329644018
Test Accuracy with RF: 0.908675799086758
Test AUC with RF:  : 0.9725516029863854
Robustess:  : 0.9881899731940824


In [70]:
importances = rf_model.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(importance_df)

                                                Variable    Importance
2324           Max(LogPatient.FiO2) where Temp <= 36.891  9.733792e-03
147                             StdDev(LogPatient.EtCO2)  8.358268e-03
12                                  Max(LogPatient.FiO2)  8.004908e-03
3275         Mean(LogPatient.EtCO2) where EtCO2 <= 31.65  5.545996e-03
4068          Mean(LogPatient.O2Sat) where O2Sat > 98.95  5.194273e-03
...                                                  ...           ...
72168  Median(LogPatient.Phosphate) where Temp in ]37... -2.255144e-21
33449  Min(LogPatient.Alkalinephos) where pH in ]7.37... -2.261570e-21
97956  Sum(LogPatient.Glucose) where Potassium in ]4.... -4.509246e-21
79311  Min(LogPatient.SaO2) where PaCO2 in ]37.52, 39... -4.510198e-21
95850  Sum(LogPatient.Alkalinephos) where Lactate in ... -9.020052e-21

[100000 rows x 2 columns]


In [71]:
count_col2 = (importance_df['Importance'] != 0).sum()
print(count_col2)

38743


In [72]:
cb_model = CatBoostClassifier(verbose=0, random_seed=42)
cb_model.fit(X_train, y_train)
y_pred = cb_model.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print(f"Train Accuracy with CB: {accuracy}")

y_pred_proba = cb_model.predict_proba(X_train)[:, 1]
auc = roc_auc_score(y_train, y_pred_proba)
print(f"Train AUC with CB:  : {auc}")
auc_train=auc

y_pred = cb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with CB: {accuracy}")

y_pred_proba = cb_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC with CB:  : {auc}")

r=auc/auc_train
print(f"Robustess:  : {r}") 


CatBoostError: bad allocation

In [64]:
importances = cb_model.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(importance_df)

                                            Variable  Importance
147                         StdDev(LogPatient.EtCO2)    4.654371
148                          StdDev(LogPatient.FiO2)    2.438507
7521     StdDev(LogPatient.FiO2) where O2Sat > 98.95    2.009562
9636          Count(LogPatient) where EtCO2 <= 25.95    1.945087
167                          StdDev(LogPatient.Temp)    1.666726
...                                              ...         ...
9990   Count(LogPatient) where WBC in ]14.91, 17.31]    0.000000
9989   Count(LogPatient) where WBC in ]13.41, 14.91]    0.000000
9988   Count(LogPatient) where WBC in ]12.23, 13.41]    0.000000
9987  Count(LogPatient) where WBC in ]11.235, 12.23]    0.000000
9984            Count(LogPatient) where WBC <= 8.575    0.000000

[10000 rows x 2 columns]


In [65]:
count_col2 = (importance_df['Importance'] != 0).sum()
print(count_col2)

2662


In [99]:
#renommage des colonnes pour XGboost
# Nombre de colonnes
n_cols = X_train.shape[1]
# Création de nouveaux noms
new_names = [f'V{i+1}' for i in range(n_cols)]
# Renommer les colonnes
X_train.columns = new_names


#renommage des colonnes pour XGboost
# Nombre de colonnes
n_cols = X_test.shape[1]
# Création de nouveaux noms
new_names = [f'V{i+1}' for i in range(n_cols)]
# Renommer les colonnes
X_test.columns = new_names

In [100]:



model_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model_xgb.fit(X_train, y_train)

y_pred = model_xgb.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print(f"Train Accuracy with CB: {accuracy}")

y_pred_proba = model_xgb.predict_proba(X_train)[:, 1]
auc = roc_auc_score(y_train, y_pred_proba)
print(f"Train AUC with CB:  : {auc}")
auc_train=auc

y_pred = model_xgb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with CB: {accuracy}")

y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC with CB:  : {auc}")

r=auc/auc_train
print(f"Robustess:  : {r}") 

C:\MesProgrammes\Python3.13.1\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:08:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Train Accuracy with CB: 1.0
Train AUC with CB:  : 1.0
Test Accuracy with CB: 0.9337899543378996
Test AUC with CB:  : 0.9832904615513312
Robustess:  : 0.9832904615513312


In [101]:
importances = model_xgb.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(importance_df)

      Variable  Importance
30419   V30420    0.085607
19243   V19244    0.039846
9055     V9056    0.038567
37759   V37760    0.037739
30474   V30475    0.016845
...        ...         ...
99979   V99980    0.000000
99980   V99981    0.000000
99981   V99982    0.000000
99982   V99983    0.000000
99983   V99984    0.000000

[100000 rows x 2 columns]


In [102]:
count_col2 = (importance_df['Importance'] != 0).sum()
print(count_col2)

1176
